In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score


In [2]:
#Loda Data Set
df = pd.read_csv(r"C:\Users\Amit Kumar Mishra\Downloads\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape

(7043, 21)

In [5]:
dups = df.duplicated()
print('Number of duplicate rows are %d' %dups.sum())

Number of duplicate rows are 0


In [6]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
# Features and target
X  = df.drop(columns=['Churn', 'customerID'])
y = df['Churn']

In [8]:
# Identify categorical and numerical columns
cat_features = X.select_dtypes(include['object']).columns
num_fetures = X.select_dtypes(include['int64', 'float64']).columns

NameError: name 'include' is not defined

In [10]:
cat_features = X.select_dtypes(include=['object']).columns
num_features = X.select_dtypes(include=['int64', 'float64']).columns

In [12]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

##✅ Key components:

ColumnTransformer:

It applies specified preprocessing steps to different columns in your dataset.

It’s especially useful when your dataset has both numerical and categorical features.

transformers=[ ... ]:
This argument defines how to process each group of features.
##✅ Details of each transformer:

Numerical features:

('num', StandardScaler(), num_features)


'num': A name you give to this transformation step.

StandardScaler(): This scales the numerical columns to have zero mean and unit variance. It’s useful when numerical features are on different scales.

num_features: A list of numerical column names you want to apply this transformation to.

Categorical features:

('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)


'cat': A name for this transformation step.

OneHotEncoder(handle_unknown='ignore'): This converts categorical variables into a one-hot encoded format (i.e., creates dummy variables). The argument handle_unknown='ignore' ensures that any unseen category during prediction will be ignored rather than causing an error.

cat_features: A list of categorical column names you want to apply this transformation to.

✅ Why this is done:

Numerical columns often need scaling because many machine learning algorithms perform better when features are on similar scales.

Categorical columns need encoding because machine learning models require numerical inputs, and one-hot encoding is a common method to represent categories numerically.

This setup ensures that both types of features are processed correctly in one step, making your code cleaner and easier to manage.

✅ Typical use:

You would later integrate this preprocessor into a pipeline with an estimator (like a classifier), for example:
This argument defines how to process each group of features.                                                                      

In [14]:
# Build pipeline with classifier
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])
                                          

In [16]:
# Train-test split
(X_train, X_test, y_train, y_test) = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [18]:
# Train model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

In [20]:
# Predictions on test data
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:,1]

In [22]:
print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))
      

              precision    recall  f1-score   support

          No       0.82      0.89      0.86      1035
         Yes       0.62      0.47      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.68      0.70      1409
weighted avg       0.77      0.78      0.77      1409

ROC-AUC Score: 0.81938179751479


In [24]:
y_pred = pipeline.predict(X_train)
y_proba = pipeline.predict_proba(X_train)[:, 1]


In [26]:
#Prediction on Traing data 
print(classification_report(y_train, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_train, y_proba))

              precision    recall  f1-score   support

          No       1.00      1.00      1.00      4139
         Yes       0.99      1.00      1.00      1495

    accuracy                           1.00      5634
   macro avg       1.00      1.00      1.00      5634
weighted avg       1.00      1.00      1.00      5634

ROC-AUC Score: 0.9998578655920799


What's happening here?

Precision and recall are nearly perfect.

Accuracy is almost 100%.

Both classes are well-predicted.

This is a red flag for overfitting.

## ✅ What is Overfitting?

Overfitting occurs when a model learns the training data too well, including noise or patterns that don't generalize to unseen data. It performs exceptionally on training (or even validation) but poorly on real-world data.

Signs of overfitting:

Extremely high accuracy on training or validation data.

Poor performance on new/unseen datasets.

The model is too complex for the amount of data available.

## ✅ How to Resolve Overfitting

Use techniques like k-fold cross-validation to ensure the model generalizes well across different data splits.

In [28]:
#Using Cross Validation (K-Fold cross validation)
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc')
print("Mean ROC-AUC:", scores.mean())
                         

Mean ROC-AUC: 0.8214886699593563


In [30]:
y_pred = pipeline.predict(X_train)
y_proba = pipeline.predict_proba(X_train)[:, 1]
print(classification_report(y_train, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_train, y_proba))


              precision    recall  f1-score   support

          No       1.00      1.00      1.00      4139
         Yes       0.99      1.00      1.00      1495

    accuracy                           1.00      5634
   macro avg       1.00      1.00      1.00      5634
weighted avg       1.00      1.00      1.00      5634

ROC-AUC Score: 0.9998578655920799


In [32]:
from sklearn.model_selection import GridSearchCV
rf = RandomForestClassifier(random_state=42)

In [34]:
# Full pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', rf)
])

In [ ]:
 

# Parameter grid
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None],
    'classifier__class_weight': ['balanced', 'balanced_subsample']
}

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42, test_size=0.2)

# Grid Search
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring='roc_auc', verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 324 candidates, totalling 1620 fits
